In [ ]:
import requests

api_url = "http://localhost:5000/generate"

schema_path = r'.\Database\employeeManagementSystemDB\schema.txt'
with open(schema_path, 'r') as f:
    schema = f.read()

question = 'who is the best performing student by each subject'

payload = {
    "model_name": "MPX0222forHF/SQL-R1-7B",
    "question": question,
    "ddl": schema,
    "dialect": "MySQL"
}

response = requests.post(api_url, json=payload)
result = response.json()

print(result["sql"])
print(f'Generated in {round(response.elapsed.total_seconds(),2)}')

WITH RankedStudents AS
  (SELECT e.employee_id,
          e.first_name,
          e.last_name,
          AVG(pm.peer_review_score) AS avg_peer_review_score,
          ROW_NUMBER() OVER (
                             ORDER BY AVG(pm.peer_review_score) DESC) AS rank
   FROM Employees e
   INNER JOIN Performance_Metrics pm ON e.employee_id = pm.employee_id
   WHERE e.employee_type = 'student'
   GROUP BY e.employee_id,
            e.first_name,
            e.last_name)
SELECT rs.first_name || ' ' || rs.last_name AS best_performing_student
FROM RankedStudents rs
WHERE rs.rank = 1;
